# Triangulation

In [1]:
import os
import pickle
import numpy as np
import matplotlib.pyplot as plt
from glob import glob
from time import time
from lib import misc, utils, app
from lib.calib import triangulate_points_fisheye

plt.style.use(os.path.join('..', 'configs', 'mplstyle.yaml'))

%load_ext autoreload
%autoreload 2

ROOT_DATA_DIR = os.path.join("..", "data")

# Reconstruction Params
Define the params in the cell below. Thereafter, run all cells

In [8]:
DATA_DIR = os.path.join(ROOT_DATA_DIR, "20250708","extrinsic_calib")
VIDEO_PATH = os.path.join(DATA_DIR)

start_frame = 1
end_frame = 999

# DLC p_cutoff - any points with likelihood < dlc_thresh are not trusted in optimisation
dlc_thresh = 0.1 # change this only if the optimisation result is unsatisfactory

print(DATA_DIR)

../data/20250708/extrinsic_calib


# Reconstruction

In [9]:
assert os.path.exists(DATA_DIR)
OUT_DIR = os.path.join(DATA_DIR, 'tri')
DLC_DIR = os.path.join(DATA_DIR, 'dlc')
assert os.path.exists(DLC_DIR)
os.makedirs(OUT_DIR, exist_ok=True)

# load video info
res, fps, tot_frames, _ = app.get_vid_info(VIDEO_PATH) # path to original videos
assert end_frame <= tot_frames, f'end_frame must be less than or equal to {tot_frames}'

start_frame = 1  # 0 based indexing
assert start_frame >= 0
N = end_frame-start_frame
print(start_frame)
print(N)

k_arr, d_arr, r_arr, t_arr, cam_res, n_cams, scene_fpath = utils.find_scene_file(DATA_DIR, verbose=False)

dlc_points_fpaths = sorted(glob(os.path.join(DLC_DIR, '*.h5')))
assert n_cams == len(dlc_points_fpaths)
    
# Load Measurement Data (pixels, likelihood)
points_2d_df = utils.load_dlc_points_as_df(dlc_points_fpaths, verbose=False)
points_2d_df = points_2d_df[points_2d_df["frame"].between(start_frame, end_frame)]
points_2d_df = points_2d_df[points_2d_df['likelihood']>dlc_thresh] # ignore points with low likelihood0

# Load Measurement Data (pixels, likelihood)
points_2d_df = utils.load_dlc_points_as_df(dlc_points_fpaths, verbose=False)


assert len(k_arr) == points_2d_df['camera'].nunique()

points_3d_df = utils.get_pairwise_3d_points_from_df(
    points_2d_df,
    k_arr, d_arr.reshape((-1,4)), r_arr, t_arr,
    triangulate_points_fisheye
)

points_3d_df['point_index'] = points_3d_df.index

1
998
Found 63898 pairwise points between camera 0 and 1
Found 63898 pairwise points between camera 1 and 2
Found 63898 pairwise points between camera 2 and 0



# Save triangulation results

In [ ]:
markers = misc.get_markers()

positions = np.full((N, len(markers), 3), np.nan)
for i, marker in enumerate(markers):
    marker_pts = points_3d_df[points_3d_df["marker"]==marker][["frame", "x", "y", "z"]].values
    for frame, *pt_3d in marker_pts:
        positions[int(frame)-start_frame-1, i] = pt_3d

app.save_tri(positions, OUT_DIR, scene_fpath, start_frame, dlc_thresh)

IndexError: index 998 is out of bounds for axis 0 with size 998

# Plot the cheetah!

In [6]:
data_fpath = os.path.join(OUT_DIR, 'tri.pickle')
app.plot_cheetah_reconstruction(data_fpath, dark_mode=True)

qt.qpa.plugin: Could not find the Qt platform plugin "wayland" in ""


Loaded extrinsics from ../data/20250806/extrinsic_calib/3_cam_scene_sba.json



/home/yayanli/anaconda3/envs/acinoset/lib/python3.7/site-packages/pyqtgraph/graphicsItems/PlotCurveItem.py:153: RuntimeWarning: All-NaN slice encountered
  b = (np.nanmin(d), np.nanmax(d))
